In [ ]:
import numpy as np
import bacco

import matplotlib.pyplot as plt
import halotools.mock_observables as ht

import os

import torch
import gpytorch

os.chdir("/cosmos_storage/home/fgmaion/MTNG-resims/scripts")
from GP_models import SMF_Model, fgas_Model

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
def get_gp_prediction(theta_original, r, gp_model):
    """
    Takes a set of 7 parameters (in original physical units) and returns 
    the predicted GP mean and variance for the full profile curve (across all mstar bins).
    """
    
    # 1. Standardize theta (using the globally defined prepare_theta function)
    theta_standardized = prepare_theta(theta_original)
    
    # 2. Construct 8D input vector (mstar and standardized theta)
    N_r_bins = len(r)
    theta_vector = np.repeat(theta_standardized[np.newaxis, :], N_r_bins, axis=0)
    
    # Full 8D input: [log10(r), theta_1_std, ..., theta_7_std]
    gp_input = np.hstack([
        r[:, np.newaxis], # (N_r_bins, 1)
        theta_vector          # (N_r_bins, 7)
    ]) 

    # 3. Predict with gpytorch
    x_star = torch.from_numpy(gp_input).float() 
    
    with torch.no_grad(), gpytorch.settings.fast_pred_var():
        predictive_dist = gp_model(x_star) 
        
    mu_pred = predictive_dist.mean.numpy()         # Predicted mean profile
    var_emulator = predictive_dist.variance.numpy() # GP intrinsic variance
    
    return mu_pred, var_emulator

In [ ]:
base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"

sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

_snap = 264
zoom_snap = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(_snap,_snap), dm_file="snapdir_{:03d}/snapshot_{:03d}".format(_snap,_snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, use_ids=False, tree_file="groups_{:03d}/subhalo_prog_{:03d}".format(_snap,_snap), numpart=4320**3)


print(zoom_snap.header['Redshift'])

In [ ]:
def get_profiles(zoom, r_bins, ih, type='dm'):

    hpos = zoom.fof['halo_pos']
    r200 = zoom.fof['halo_r200c']

    _hpos = np.array([hpos[ih]])
    _r200 = np.array([r200[ih]])

    if type=='dm':
        _pos = zoom.dm['pos']
        _mass = np.ones_like(_pos[:,0])  * zoom_snap.header['ParticleMass'] * 1e10
    elif type=='gas':
        _pos = zoom.gas['pos']
        _mass = zoom.gas['mass'] * 1e10
    elif type=='stars':
        _pos = zoom.stars['pos']
        _mass = zoom.stars['mass'] * 1e10
    elif type=='bh':
        _pos = zoom.bh['pos']
        _mass = zoom.bh['mass'] * 1e10

    mask = (_pos[:,0] > _hpos[0,0]-10) & (_pos[:,0] < _hpos[0,0]+10) & \
           (_pos[:,1] > _hpos[0,1]-10) & (_pos[:,1] < _hpos[0,1]+10) & \
           (_pos[:,2] > _hpos[0,2]-10) & (_pos[:,2] < _hpos[0,2]+10)

    y, c = ht.radial_profile_3d(_hpos, _pos[mask], _mass[mask], return_counts=True, rbins_absolute=r_bins)
    rr = 10**((np.log10(r_bins[1:])+np.log10(r_bins[:-1]))*0.5)
    #rr = (rbins[1:]+rbins[:-1])*0.5
    volume = 4 * np.pi / 3 *(r_bins[1:]**3 - r_bins[:-1]**3)

    dens1 = y * c / volume
    x1 = rr

    return x1, dens1

In [ ]:
r_dm, rho_dm = get_profiles(zoom_snap, np.logspace(-2, 1, 20), 0, type='dm')
r_gas, rho_gas = get_profiles(zoom_snap, np.logspace(-2, 1, 20), 0, type='gas')
r_stars, rho_stars = get_profiles(zoom_snap, np.logspace(-2, 1, 20), 0, type='stars')
r_bh, rho_bh = get_profiles(zoom_snap, np.logspace(-2, 1, 20), 0, type='bh')

In [ ]:
name_list = ["LH_{:d}".format(i) for i in range(30)] + ["fiducial"]

prof_m14 = {}

for i in range(len(name_list)):
    prof_m14[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/profiles/prof_clusters_{}.npy".format(name_list[i]), allow_pickle=True).item()

In [ ]:
## Load the emulator

model_prof = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_model_rho_groups.pth")
likelihood_prof = torch.load("/cosmos_storage/home/fgmaion/MTNG-resims/gp_train_results/full_likelihood_rho_groups.pth")

model_prof.eval()
likelihood_prof.eval()

In [ ]:
f_b

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('linear')

# for i in range(1):#range(len(name_list)):
    # ax.plot(np.mean(prof_m14[name_list[i]]['r']['dm'], axis=0), np.mean(prof_m14[name_list[i]]['rho']['dm'], axis=0), ls='-', color="C"+str(2), label='Gas Profile')
for i in range(2):#range(len(name_list)):
    ax.plot(np.mean(prof_m14[name_list[i]]['r']['gas'], axis=0), np.mean(prof_m14[name_list[i]]['rho']['gas'], axis=0) / f_b / np.mean(prof_m14[name_list[i]]['rho']['dm'], axis=0), ls='-', color="C"+str(0), label='Gas Profile')
for i in range(2):#range(len(name_list)):
    ax.plot(np.mean(prof_m14[name_list[i]]['r']['stars'], axis=0), np.mean(prof_m14[name_list[i]]['rho']['stars'], axis=0) / f_b / np.mean(prof_m14[name_list[i]]['rho']['dm'], axis=0), color="C"+str(3), label='Stellar Profile')

ax.set_xlabel("$r/r_{500,c}$", fontsize=16)
ax.set_ylabel("$\\rho/f_b \\rho_{DM}$", fontsize=16)
ax.axhline(1, ls='--', color='k', label="Cosmic Baryon Fraction")

ax.set_title("Clusters $M_h\\in[10^{14},...] M_\\odot$", fontsize=16)
ax.legend()

In [ ]:
name_list = ["LH_{:d}".format(i) for i in range(30)] + ["fiducial"]

prof_groups = {}

for i in range(len(name_list)):
    prof_groups[name_list[i]] = np.load("/cosmos_storage/home/fgmaion/MTNG-resims/results/profiles/prof_groups_{}.npy".format(name_list[i]), allow_pickle=True).item()

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale('log')
ax.set_yscale('linear')

f_b = zoom_snap.Cosmology.pars['omega_baryon'] / zoom_snap.Cosmology.pars['omega_matter']

for i in range(len(name_list)):
    ax.plot(np.mean(prof_groups[name_list[i]]['r']['gas'], axis=0), np.mean(prof_groups[name_list[i]]['rho']['gas'], axis=0) / f_b / np.mean(prof_groups[name_list[i]]['rho']['dm'], axis=0),\
        ls='-', color="C"+str(0), label='Gas Profile')
for i in range(len(name_list)):
    ax.plot(np.mean(prof_groups[name_list[i]]['r']['stars'], axis=0), np.mean(prof_groups[name_list[i]]['rho']['stars'], axis=0) / f_b / np.mean(prof_groups[name_list[i]]['rho']['dm'], axis=0),\
        color="C"+str(3), label='Stellar Profile')

ax.set_xlabel("$r/r_{200,c}$", fontsize=16)
ax.set_ylabel("$\\rho$ [$M_\\odot / \\mathrm{kpc}^3$]", fontsize=16)

ax.set_title("Clusters $M_h\\in[10^{13},10^{13.5}] M_\\odot$", fontsize=16)
# ax.legend()

In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(5.5,5))

ax.set_xscale("log")
ax.set_yscale("log")

fb = zoom_snap.Cosmology.pars['omega_baryon'] / zoom_snap.Cosmology.pars['omega_matter']
ax.plot(r_dm/r200[0], rho_dm * fb, label='DM', color='k')
ax.plot(r_gas/r200[0], rho_gas, label='Gas', color='C3')
ax.plot(r_stars/r200[0], rho_stars, label='Stars', color='C0')
ax.plot(r_bh/r200[0], rho_bh, label='BH', color='C2')

ax.set_ylabel("$\\rho[M_\\odot/[Mpc/h]^3]$")
ax.set_xlabel("$R/R_{200,c}$")